# 02 - Cleaning and Merge

**Goal:** turn the raw, station-level 2016 and 2021 results into a single **ward-level panel** with turnout figures for both years, ready for feature engineering and modelling.

**Steps in this notebook:**
1. Collapse station-level rows to one row per ward (fixing the repeated-field and multi-ballot-type issues in the raw data)
2. Check ward-ID overlap between 2016 and 2021
3. Merge the two years into one panel
4. Investigate and handle outlier/implausible turnout values
5. Save the final clean panel to `data/processed/ward_panel.csv`

In [1]:
import glob
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
def load_province_files(year_dir: str) -> pd.DataFrame:
    files = sorted(glob.glob(f"{year_dir}/*.csv"))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


df_2016 = load_province_files("../data/raw/LGE2016")
df_2021 = load_province_files("../data/raw/LGE2021")

## Step 1 - Collapse to ward level

Two things have to be handled carefully here, both found by manual inspection of the raw data:

- **`RegisteredVoters` and `SpoiltVotes` are repeated on every party row for a station.** Naively summing them across party rows would wildly overcount. We take these with `.agg(..., "first")` at the station level, once we've grouped down to one row per station.
- **`BallotType` has three values: `Ward`, `PR`, and `DC 40%`.** These are three *separate* ballots a voter can cast, not subsets of one ballot - summing across all three would overcount votes, and would do so *inconsistently* (metros don't have a district-council ballot, local municipalities do). We filter to the **`Ward`** ballot only, since every registered voter casts exactly one ward vote regardless of municipality type, making it the most consistent proxy for turnout.

In [3]:
def to_ward_level(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse voting-station x party rows to one row per ward.

    Filters to the 'Ward' ballot type before aggregating, since PR and
    DC 40% ballots would otherwise inflate (and inconsistently inflate,
    across municipality types) the vote totals.
    """
    df = df[df["BallotType"] == "Ward"].copy()

    # Collapse to one row per voting station first, since RegisteredVoters and SpoiltVotes are duplicated across every party row for a station.
    station = df.groupby(
        ["Province", "Municipality", "Ward", "VotingDistrict"], as_index=False
    ).agg(
        RegisteredVoters=("RegisteredVoters", "first"),
        SpoiltVotes=("SpoiltVotes", "first"),
        TotalValidVotes=("TotalValidVotes", "sum"),  # sum across ward candidates
    )

    # Then aggregate stations up to ward level.
    ward = station.groupby(
        ["Province", "Municipality", "Ward"], as_index=False
    ).agg(
        RegisteredVoters=("RegisteredVoters", "sum"),
        SpoiltVotes=("SpoiltVotes", "sum"),
        TotalValidVotes=("TotalValidVotes", "sum"),
    )
    ward["VotesCast"] = ward["TotalValidVotes"] + ward["SpoiltVotes"]
    ward["Turnout"] = ward["VotesCast"] / ward["RegisteredVoters"]
    return ward


ward_2016 = to_ward_level(df_2016)
ward_2021 = to_ward_level(df_2021)

print("2016 wards:", ward_2016["Ward"].nunique())
print("2021 wards:", ward_2021["Ward"].nunique())


2016 wards: 4392
2021 wards: 4468


## Step 2 - Ward-ID overlap check

This is the single riskiest step in the pipeline: does the same `Ward` code refer to the same physical ward in both years, or has municipal demarcation shifted things around?

In [4]:
overlap = set(ward_2016["Ward"]) & set(ward_2021["Ward"])
print(f"2016 wards: {ward_2016['Ward'].nunique()}, "
      f"2021 wards: {ward_2021['Ward'].nunique()}, "
      f"overlap: {len(overlap)}")


2016 wards: 4392, 2021 wards: 4468, overlap: 4349


**Result: 4,349 of 4,392 2016 wards (99.0%) and 4,349 of 4,468 2021 wards (97.3%) matched.** The small non-overlap is consistent with genuine municipal demarcation changes (wards split, merged, or renumbered) between the two election cycles - not a systemic ID or parsing problem. An inner join on `Ward` is a defensible choice for the modelling panel.

In [5]:
# Spot-check the non-matching wards look like real, well-formed ward codes (i.e. genuine demarcation differences), not parsing artifacts.
missing_2021 = set(ward_2016["Ward"]) - set(ward_2021["Ward"])
missing_2016 = set(ward_2021["Ward"]) - set(ward_2016["Ward"])

print(ward_2016[ward_2016["Ward"].isin(missing_2021)][["Province", "Municipality", "Ward"]].head(10))
print(ward_2021[ward_2021["Ward"].isin(missing_2016)][["Province", "Municipality", "Ward"]].head(10))


          Province                    Municipality           Ward
62    Eastern Cape                EC101 - Camdeboo  Ward 21001013
63    Eastern Cape                EC101 - Camdeboo  Ward 21001014
915     Free State                 FS201 - Moqhaka  Ward 42001023
1266       Gauteng  GT485 - Randfontein/Westonaria  Ward 74205001
1267       Gauteng  GT485 - Randfontein/Westonaria  Ward 74205002
1268       Gauteng  GT485 - Randfontein/Westonaria  Ward 74205003
1269       Gauteng  GT485 - Randfontein/Westonaria  Ward 74205004
1270       Gauteng  GT485 - Randfontein/Westonaria  Ward 74205005
1271       Gauteng  GT485 - Randfontein/Westonaria  Ward 74205006
1272       Gauteng  GT485 - Randfontein/Westonaria  Ward 74205007
         Province                       Municipality           Ward
184  Eastern Cape                     EC122 - Mnquma  Ward 21202032
543  Eastern Cape     EC157 - King Sabata Dalindyebo  Ward 21507037
570  Eastern Cape                  EC441 - Matatiele  Ward 24401027
59

## Step 3 - Merge into a ward-level panel

**Note:** we merge on `["Province", "Ward"]` only, *not* `Municipality`. Some municipalities were renamed or merged between 2016 and 2021 (e.g. `EC101 - Camdeboo` → `EC101 - Dr. Beyers Naude`) while the municipality **code** and ward code stayed stable - merging on the full municipality name string would have silently dropped ~730 valid ward matches.

In [6]:
ward_panel = ward_2016.merge(
    ward_2021, on=["Province", "Ward"], suffixes=("_2016", "_2021")
)
print(ward_panel.shape)
ward_panel.head()

(4349, 14)


,Province,Municipality_2016,Ward,RegisteredVoters_2016,SpoiltVotes_2016,TotalValidVotes_2016,VotesCast_2016,Turnout_2016,Municipality_2021,RegisteredVoters_2021,SpoiltVotes_2021,TotalValidVotes_2021,VotesCast_2021,Turnout_2021
0,Eastern Cape,BUF - Buffalo City,Ward 29200001,8851,80,4973,5053,0.570896,BUF - Buffalo City,9589,37,3803,3840,0.400459
1,Eastern Cape,BUF - Buffalo City,Ward 29200002,7794,74,3562,3636,0.466513,BUF - Buffalo City,7655,82,3299,3381,0.441672
2,Eastern Cape,BUF - Buffalo City,Ward 29200003,8118,34,3351,3385,0.416975,BUF - Buffalo City,9961,46,3326,3372,0.338520
3,Eastern Cape,BUF - Buffalo City,Ward 29200004,9175,25,6161,6186,0.674223,BUF - Buffalo City,9327,60,4524,4584,0.491476
4,Eastern Cape,BUF - Buffalo City,Ward 29200005,9228,102,5069,5171,0.560360,BUF - Buffalo City,8732,20,3506,3526,0.403802


In [7]:
# Confirm the municipality-name mismatches are just renames/mergers, not evidence of a bad join (same ward, different municipality label).
mismatched_muni = ward_panel[
    ward_panel["Municipality_2016"] != ward_panel["Municipality_2021"]
]
print(f"{len(mismatched_muni)} wards have a municipality name that changed between years")
mismatched_muni[["Province", "Municipality_2016", "Municipality_2021", "Ward"]].head(10)


730 wards have a municipality name that changed between years


,Province,Municipality_2016,Municipality_2021,Ward
50,Eastern Cape,EC101 - Camdeboo,EC101 - Dr. Beyers Naude,Ward 21001001
51,Eastern Cape,EC101 - Camdeboo,EC101 - Dr. Beyers Naude,Ward 21001002
52,Eastern Cape,EC101 - Camdeboo,EC101 - Dr. Beyers Naude,Ward 21001003
53,Eastern Cape,EC101 - Camdeboo,EC101 - Dr. Beyers Naude,Ward 21001004
54,Eastern Cape,EC101 - Camdeboo,EC101 - Dr. Beyers Naude,Ward 21001005
55,Eastern Cape,EC101 - Camdeboo,EC101 - Dr. Beyers Naude,Ward 21001006
56,Eastern Cape,EC101 - Camdeboo,EC101 - Dr. Beyers Naude,Ward 21001007
57,Eastern Cape,EC101 - Camdeboo,EC101 - Dr. Beyers Naude,Ward 21001008
58,Eastern Cape,EC101 - Camdeboo,EC101 - Dr. Beyers Naude,Ward 21001009
59,Eastern Cape,EC101 - Camdeboo,EC101 - Dr. Beyers Naude,Ward 21001010


## Step 4 - Outlier investigation

Turnout should sit somewhere in roughly [0.1, 1.0]. Anything outside that range needs an explanation before it goes into a training set - each flagged case below was traced back to its raw rows rather than dropped blind.

In [8]:
outliers = ward_panel[
    (ward_panel["Turnout_2016"] > 1.0) | (ward_panel["Turnout_2016"] < 0.05)
]
outliers[["Province", "Municipality_2016", "Ward", "RegisteredVoters_2016", "VotesCast_2016", "Turnout_2016"]]

,Province,Municipality_2016,Ward,RegisteredVoters_2016,VotesCast_2016,Turnout_2016
2601,Limpopo,LIM345 - New Municipality,Ward 93405007,3587,91,0.025369
2602,Limpopo,LIM345 - New Municipality,Ward 93405008,3662,56,0.015292
2607,Limpopo,LIM345 - New Municipality,Ward 93405013,3866,108,0.027936
2608,Limpopo,LIM345 - New Municipality,Ward 93405014,4468,124,0.027753
3694,North West,NW403 - Matlosana,Ward 64003036,4065,5000,1.230012


**Finding 1 - 4 wards in Limpopo (`LIM345 - New Municipality`), turnout ~0.02–0.03:**

Traced to the raw data: the municipality is genuinely recorded as "New Municipality" in the source file, with no duplicate rows. `LIM345` corresponds to a municipal demarcation dispute that was unresolved at the time of the 2016 election, which disrupted voting in that area. This is a real-world event reflected faithfully in the data, not a pipeline bug - but it makes these 4 wards unrepresentative of normal voter behaviour, so they are excluded from the modelling panel.

**Finding 2 - 1 ward in North West (Matlosana, `Ward 64003036`), turnout 1.23:**

Traced to the raw data: registered voters (4,065) matched the sum of two voting stations exactly (1,424 + 2,641), ruling out duplication on that side. But one station ("DG MINISTRIES") recorded 4,268 votes against only 2,641 registered voters there - more votes than registered voters, which is impossible. This points to a source-data error in the IEC export (likely a vote/station mismatch) rather than anything fixable from the cleaned data. This ward is excluded.

In [9]:
exclude_wards = [
    "Ward 93405007", "Ward 93405008", "Ward 93405013", "Ward 93405014",  # LIM345 demarcation dispute
    "Ward 64003036",  # Matlosana - implausible votes vs. registered voters, likely source-data error
]

ward_panel_clean = ward_panel[~ward_panel["Ward"].isin(exclude_wards)].copy()
print(ward_panel_clean.shape)
ward_panel_clean[["Turnout_2016", "Turnout_2021"]].describe()


(4344, 14)


,Turnout_2016,Turnout_2021
count,4344.000000,4344.000000
mean,0.571988,0.467693
std,0.078446,0.086488
min,0.115321,0.119934
25%,0.521763,0.410007
50%,0.570211,0.462686
75%,0.620414,0.522558
max,0.857858,0.855718


Turnout now ranges sensibly between ~0.12 and ~0.86 for both years, and the panel means (57.2% in 2016, 46.8% in 2021) line up closely with South Africa's reported national LGE turnout for those years - a good independent sanity check that the ballot-type filtering and aggregation logic are correct.

## Step 5 - Add a stable municipality code and save the panel

In [10]:
# Municipality codes (e.g. 'EC101') stayed stable across renames/mergers,
# unlike the full municipality name strings - use the code as the
# canonical identifier going forward, taken from the more current (2021) label.
ward_panel_clean["MunicipalityCode"] = (
    ward_panel_clean["Municipality_2021"].str.split(" - ").str[0]
)

output_cols = [
    "Province", "Ward", "MunicipalityCode",
    "Municipality_2016", "Municipality_2021",
    "RegisteredVoters_2016", "VotesCast_2016", "Turnout_2016",
    "RegisteredVoters_2021", "VotesCast_2021", "Turnout_2021",
]
ward_panel_final = ward_panel_clean[output_cols]
ward_panel_final.to_csv("../data/processed/ward_panel.csv", index=False)
print(f"Saved {len(ward_panel_final)} wards to data/processed/ward_panel.csv")


Saved 4344 wards to data/processed/ward_panel.csv


## Handoff notes for feature engineering / modelling

**File:** `data/processed/ward_panel.csv` - one row per ward, 4,344 wards.

**Columns:** `Province`, `Ward`, `MunicipalityCode`, `Municipality_2016`, `Municipality_2021`, `RegisteredVoters_2016`, `VotesCast_2016`, `Turnout_2016`, `RegisteredVoters_2021`, `VotesCast_2021`, `Turnout_2021`.

**Data limitations to carry forward into the write-up:**
- Ward matching between years used exact `Ward` code + `Province`. 99.0% of 2016 wards and 97.3% of 2021 wards matched; the remainder were excluded from the panel and likely reflect genuine municipal demarcation changes between cycles.
- Turnout is computed from the **Ward ballot only** (not PR or DC 40%), for consistency across metro and local municipality types.
- 4 wards (Limpopo, LIM345) were excluded due to a documented municipal demarcation dispute that disrupted 2016 voting.
- 1 ward (North West, Matlosana) was excluded due to an apparent source-data error (votes exceeding registered voters at one station).
- Some municipalities were renamed/merged between 2016 and 2021 (e.g. Camdeboo → Dr. Beyers Naude); municipality **codes** were used as the stable identifier rather than names.